In [1]:
# Si trabajas en Google Colab, revisa esta guía de ejecución de notebooks:
# https://docs.pytorch.org/tutorials/beginner/colab
# Activa la visualización de gráficos dentro del propio notebook.
%matplotlib inline

Creación de un Dataloader a parti de un fichero .cvs - TITANIC
==============================================================

Vamos a crear un Dataloader a partir del fichero titanic3.csv, ya que será habitual que tengamos
que crear conjunto de datos con los que entrenar y validar una red neuronal, a partir de información
almacenada en un fichero .csv

In [1]:
import pandas as pd

# Obtenemos un DataFrame de pandas leyendo el archivo CSV del dataset del Titanic.
titanic = pd.read_csv('titanic3.csv')

print(f'Tipo de dato donde se recoge el csv: {type(titanic)}')

print(f'Columnas del DataFrame de Titanic:\n{titanic.columns}')

# Muestra de las primeras filas del DataFrame para verificar que se ha cargado correctamente.
print(f'Primeras filas del DataFrame de Titanic:\n{titanic.head()}')

Tipo de dato donde se recoge el csv: <class 'pandas.core.frame.DataFrame'>
Columnas del DataFrame de Titanic:
Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='object')
Primeras filas del DataFrame de Titanic:
   pclass  survived                                             name     sex  \
0       1         1                    Allen, Miss. Elisabeth Walton  female   
1       1         1                   Allison, Master. Hudson Trevor    male   
2       1         0                     Allison, Miss. Helen Loraine  female   
3       1         0             Allison, Mr. Hudson Joshua Creighton    male   
4       1         0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   

       age  sibsp  parch  ticket      fare    cabin embarked boat   body  \
0  29.0000      0      0   24160  211.3375       B5        S    2    NaN   
1   0.9167      1      2  113781  151.5500  C22 C

Aquí tenemos una descripción de lo que significan cada una de las variables:

```
pclass          Passenger Class
                (1 = 1st; 2 = 2nd; 3 = 3rd)
survival        Survival
                (0 = No; 1 = Yes)
name            Name
sex             Sex
age             Age
sibsp           Number of Siblings/Spouses Aboard
parch           Number of Parents/Children Aboard
ticket          Ticket Number
fare            Passenger Fare
cabin           Cabin
embarked        Port of Embarkation
                (C = Cherbourg; Q = Queenstown; S = Southampton)
boat            Lifeboat
body            Body Identification Number
home.dest       Home/Destination
```

Parece que las variables `name`, `sex`, `cabin`, `embarked`, `boat`, `body` y `homedest` son candidatas a ser variables categóricas, mientras que el resto parecen variables numéricas. 

Una vez tenemos el .csv cargado en un DataFrame de Pandas, podemos hacer operaciones sobre el antes de crear un conjunto de  entrenamientos y test:
1) Voy a pasar la columna de supervivencia al final para una interpretación más clara. Va a ser un problema de clasificación de "Superviviente" o "No superviviente".

In [2]:
# Pasamos la columna 'Survived' a la ultima posición del Dataframe.

# Extraccion de la columna 'survided'
col_to_move = titanic.pop('survived')

# La añadimos ahora al final del DataFrame.
titanic.insert(len(titanic.columns), 'survived', col_to_move)

# Comprobamos ahora que la columna 'survived' está al final del DataFrame.
print(f'Primeras filas del DataFrame de Titanic:\n{titanic.head()}')    

Primeras filas del DataFrame de Titanic:
   pclass                                             name     sex      age  \
0       1                    Allen, Miss. Elisabeth Walton  female  29.0000   
1       1                   Allison, Master. Hudson Trevor    male   0.9167   
2       1                     Allison, Miss. Helen Loraine  female   2.0000   
3       1             Allison, Mr. Hudson Joshua Creighton    male  30.0000   
4       1  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female  25.0000   

   sibsp  parch  ticket      fare    cabin embarked boat   body  \
0      0      0   24160  211.3375       B5        S    2    NaN   
1      1      2  113781  151.5500  C22 C26        S   11    NaN   
2      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest  survived  
0                     St Louis, MO

2) Vamos ahora a quedarnos solo con las características que nos interesen. 

    Se podrían hacer una selección de características, pero para abreviar cogeremos directamente algunas.

    Podemos descartar directamente las columnas "boat" y "body" ya que está directamente relacionadas con que el pasajero sobreviviese. "name"  es (probablemente) único para cada persona y por tanto no es informativo. "embarked tampoco parece que puede tener relevancia. Lo mismo con "home.dest".

    Vamos a quedarnos con: "pclass", "sex", "age", "sibsp", "parch", "ticket", "fare" y "survived" como características.

In [3]:
# Extraermos un subconjunto de columnas. Para ello hay que indicarlas entre corchetes, y se pueden indicar por su nombre o por su posición (índice). Aquí lo hacemos por nombre.
titanic_with_features = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'survived']]

# Imprimios el nuevo conjunto de datos
print(f'Primeras filas del subconjunto de columnas seleccionadas:\n{titanic_with_features.head()}')

Primeras filas del subconjunto de columnas seleccionadas:
   pclass     sex      age  sibsp  parch      fare  survived
0       1  female  29.0000      0      0  211.3375         1
1       1    male   0.9167      1      2  151.5500         1
2       1  female   2.0000      1      2  151.5500         0
3       1    male  30.0000      1      2  151.5500         0
4       1  female  25.0000      1      2  151.5500         0


3) Vamos a pasar a númerico la columna "sex". Esto simula el preprocesamiento del dataset.

In [4]:
# Convertimos "sex" a numérico con codificación one-hot (1 de J)
titanic_with_features = pd.get_dummies(titanic_with_features,columns=["sex"],dtype=int)
print(f'Primeras filas del subconjunto de columnas seleccionadas:\n{titanic_with_features.head()}')

# Las nuevas columnas se han añadido al final del DataFrame, después de "survived".
# Pasamos a poner otra vez "survived" al final (por claridad)
col_survived = titanic_with_features.pop("survived")
titanic_with_features.insert(len(titanic_with_features.columns), "survived", col_survived)
print(f'Primeras filas del subconjunto de columnas seleccionadas:\n{titanic_with_features.head()}')

Primeras filas del subconjunto de columnas seleccionadas:
   pclass      age  sibsp  parch      fare  survived  sex_female  sex_male
0       1  29.0000      0      0  211.3375         1           1         0
1       1   0.9167      1      2  151.5500         1           0         1
2       1   2.0000      1      2  151.5500         0           1         0
3       1  30.0000      1      2  151.5500         0           0         1
4       1  25.0000      1      2  151.5500         0           1         0
Primeras filas del subconjunto de columnas seleccionadas:
   pclass      age  sibsp  parch      fare  sex_female  sex_male  survived
0       1  29.0000      0      0  211.3375           1         0         1
1       1   0.9167      1      2  151.5500           0         1         1
2       1   2.0000      1      2  151.5500           1         0         0
3       1  30.0000      1      2  151.5500           0         1         0
4       1  25.0000      1      2  151.5500           1     

--- FORMA 1 ---

4) Una vez tenemos el Dataframe que deseamos a partir del .csv, dividimos el conjunto de datos en training, validation y test. 

Pasos:

    - 1) Separar en Dataframe X (características) y Dataframe y (etiquetas).
    - 2) Pasar el Dataframe X y la Serie y a numpy con to_numpy(). DataFrame -> numpy
    - 3) Partir en conjuntos estratificados de entrenamiento, validación y test, con train_test_split().
    - 4) Convertir de numpy -> tensor con torch.from_numpy() o torch.Tensor()
    - 5) Convertir de tensor -> TensorDataset.
    - 6) Crear los DataLoader (train/val/test), TensorDataset -> DataLoader

In [12]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 1) Separar características y etiqueta
# ======================================
# Crea un nuevo DataFrame X con todas las columnas de "titanic_with_features" excepto "survived":
X = titanic_with_features.drop(columns=["survived"])
print(f'Tipo de dato de X: {type(X)}')

# Extraemos la columna "survived" como vector de etiquetas y lo guardamos en "y".
# Usamos esto en vez de: 
# "y = titanic_with_features.pop('survived')"
# para no modificar el Dataframe original.
y = titanic_with_features["survived"]
print(f'Tipo de dato de y: {type(y)}\n')

# 2) Pasar Dataframe -> NumPy
# ======================================
X = X.to_numpy(dtype="float32")
print(f'Tipo de dato de X después de to_numpy: {type(X)}')

y = y.to_numpy(dtype="float32")  
print(f'Tipo de dato de y después de to_numpy: {type(y)}\n')

# 3) Split estratificado: 70% train, 15% val, 15% test
# ======================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=0
)
print(f'Tipo de dato de X_train: {type(X_train)}')
print(f'Tipo de dato de y_train: {type(y_train)}')
print(f'Tipo de dato de X_val: {type(X_val)}')
print(f'Tipo de dato de y_val: {type(y_val)}')
print(f'Tipo de dato de X_test: {type(X_test)}')
print(f'Tipo de dato de y_test: {type(y_test)}\n')

# 4) NumPy -> tensores 
#===========================================
X_train_t = torch.from_numpy(X_train)                    # float32
y_train_t = torch.from_numpy(y_train).unsqueeze(1)       # [N, 1] para BCE

X_val_t = torch.from_numpy(X_val)
y_val_t = torch.from_numpy(y_val).unsqueeze(1)

X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test).unsqueeze(1)

print(f'Tipo de dato de X_train_tensor: {type(X_train_t)}')
print(f'Tipo de dato de y_train_tensor: {type(y_train_t)}')
print(f'Tipo de dato de X_val_tensor: {type(X_val_t)}')
print(f'Tipo de dato de y_val_tensor: {type(y_val_t)}')
print(f'Tipo de dato de X_test_tensor: {type(X_test_t)}')
print(f'Tipo de dato de y_test_tensor: {type(y_test_t)}\n')

# 5) De Tensor -> TensorDataset
#============================================
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)
print(f'Tipo de dato de train_ds: {type(train_ds)}')
print(f'Tipo de dato de val_ds: {type(val_ds)}')
print(f'Tipo de dato de test_ds: {type(test_ds)}\n')

# 6)De TensorDataset -> DataLoader
# ============================================
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}\n")

# Obtener una muestra del primer patrón del dataset con train_loader
first_batch = next(iter(train_loader))
X_batch, y_batch = first_batch

print(f"Primer batch de características (X):\n{X_batch[0]}")
print(f"Primera etiqueta (y): {y_batch[0]}")

Tipo de dato de X: <class 'pandas.core.frame.DataFrame'>
Tipo de dato de y: <class 'pandas.core.series.Series'>

Tipo de dato de X después de to_numpy: <class 'numpy.ndarray'>
Tipo de dato de y después de to_numpy: <class 'numpy.ndarray'>

Tipo de dato de X_train: <class 'numpy.ndarray'>
Tipo de dato de y_train: <class 'numpy.ndarray'>
Tipo de dato de X_val: <class 'numpy.ndarray'>
Tipo de dato de y_val: <class 'numpy.ndarray'>
Tipo de dato de X_test: <class 'numpy.ndarray'>
Tipo de dato de y_test: <class 'numpy.ndarray'>

Tipo de dato de X_train_tensor: <class 'torch.Tensor'>
Tipo de dato de y_train_tensor: <class 'torch.Tensor'>
Tipo de dato de X_val_tensor: <class 'torch.Tensor'>
Tipo de dato de y_val_tensor: <class 'torch.Tensor'>
Tipo de dato de X_test_tensor: <class 'torch.Tensor'>
Tipo de dato de y_test_tensor: <class 'torch.Tensor'>

Tipo de dato de train_ds: <class 'torch.utils.data.dataset.TensorDataset'>
Tipo de dato de val_ds: <class 'torch.utils.data.dataset.TensorDataset'

--- FORMA 2 ---

4) Una vez tenemos el Dataframe que deseamos a partir del .csv, dividimos el conjunto de datos en training, validation y test. 

Pasos:

    - 1) Separar en Dataframe X (características) y Dataframe y (etiquetas).
    - 2) Pasar el Dataframe X y la Serie y a numpy con to_numpy(). DataFrame -> numpy
    - 3) Convertir de numpy -> tensor con torch.from_numpy() o torch.Tensor()
    - 4) Convertir de los dos Tensor "X" e "y" a un solo -> TensorDataset.
    - 5) Partir en conjuntos estratificados de entrenamiento, validación y test, con train_test_split().
    - 6) Crear los DataLoader (train/val/test), TensorDataset -> DataLoader

In [13]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 1) Separar características y etiqueta
# ======================================
# Crea un nuevo DataFrame X con todas las columnas de "titanic_with_features" excepto "survived":
X = titanic_with_features.drop(columns=["survived"])
print(f'Tipo de dato de X: {type(X)}')

# Extraemos la columna "survived" como vector de etiquetas y lo guardamos en "y".
# Usamos esto en vez de:
# "y = titanic_with_features.pop('survived')"
# para no modificar el Dataframe original.
y = titanic_with_features["survived"]
print(f'Tipo de dato de y: {type(y)}\n')

# 2) Pasar DataFrame -> NumPy
# ======================================
X = X.to_numpy(dtype="float32")
print(f'Tipo de dato de X después de to_numpy: {type(X)}')

y = y.to_numpy(dtype="float32")
print(f'Tipo de dato de y después de to_numpy: {type(y)}\n')

# 3) NumPy -> Tensor
# ===========================================
X = torch.tensor(X, dtype=torch.float32)
print(f'Tipo de dato de X después de torch.tensor: {type(X)}')

y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
print(f'Tipo de dato de y después de torch.tensor: {type(y)}\n')

# 4) De Tensor -> TensorDataset
# ===========================================
data = TensorDataset(X, y)
print(f'Tipo de dato de data: {type(data)}\n')

# 5) Split estratificado simple (70% train, 15% val, 15% test) desde 'data'
# ======================================
X_all, y_all = data.tensors
print(f'Tipo de dato de X_all: {type(X_all)}')
print(f'Tipo de dato de y_all: {type(y_all)}\n\n')

# Se hace una partición estratificada pero solo con dos conjuntos,
# el de entrenamiento y el de test. Posteriormente habrá que
# partir el de test otra vez para obtener validación y test.

#        --- OJO ----
#     y_all.squeeze(1)
#
#        - Convierte y_all de forma [N, 1] a [N].
#         -Es decir, quita la dimensión extra de tamaño 1
#          para dejar un vector de etiquetas plano.
X_train, X_test_temp, y_train, y_test_temp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all.squeeze(1), random_state=0
)

# Ahora cogemos el conjunto de test y lo partimos a su vez en 
# validación y test, también de forma estratificada. 
# Al poner test_size=0.50, se divide el conjunto de test_temp
# en dos partes iguales: una para validación y otra para test.
X_val, X_test, y_val, y_test = train_test_split(
    X_test_temp, y_test_temp, test_size=0.50, stratify=y_test_temp.squeeze(1), random_state=42
)

print(f'Tipo de dato de X_train: {type(X_train)}')
print(f'Tipo de dato de y_train: {type(y_train)}')
print(f'Tipo de dato de X_val: {type(X_val)}')
print(f'Tipo de dato de y_val: {type(y_val)}')
print(f'Tipo de dato de X_test: {type(X_test)}')
print(f'Tipo de dato de y_test: {type(y_test)}\n')

# 6) De Tensor -> TensorDataset
# ============================================
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
test_ds = TensorDataset(X_test, y_test)

print(f'Tipo de dato de train_ds: {type(train_ds)}')
print(f'Tipo de dato de val_ds: {type(val_ds)}')
print(f'Tipo de dato de test_ds: {type(test_ds)}\n')

# 7) De TensorDataset -> DataLoader
# ============================================
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}\n")

# Obtener una muestra del primer patrón del dataset con train_loader
first_batch = next(iter(train_loader))
X_batch, y_batch = first_batch

print(f"Primer batch de características (X):\n{X_batch[0]}")
print(f"Primera etiqueta (y): {y_batch[0]}")

Tipo de dato de X: <class 'pandas.core.frame.DataFrame'>
Tipo de dato de y: <class 'pandas.core.series.Series'>

Tipo de dato de X después de to_numpy: <class 'numpy.ndarray'>
Tipo de dato de y después de to_numpy: <class 'numpy.ndarray'>

Tipo de dato de X después de torch.tensor: <class 'torch.Tensor'>
Tipo de dato de y después de torch.tensor: <class 'torch.Tensor'>

Tipo de dato de data: <class 'torch.utils.data.dataset.TensorDataset'>

Tipo de dato de X_all: <class 'torch.Tensor'>
Tipo de dato de y_all: <class 'torch.Tensor'>


Tipo de dato de X_train: <class 'torch.Tensor'>
Tipo de dato de y_train: <class 'torch.Tensor'>
Tipo de dato de X_val: <class 'torch.Tensor'>
Tipo de dato de y_val: <class 'torch.Tensor'>
Tipo de dato de X_test: <class 'torch.Tensor'>
Tipo de dato de y_test: <class 'torch.Tensor'>

Tipo de dato de train_ds: <class 'torch.utils.data.dataset.TensorDataset'>
Tipo de dato de val_ds: <class 'torch.utils.data.dataset.TensorDataset'>
Tipo de dato de test_ds: <clas